# 04 — Análise dos resultados

Define métricas em chunks, hipervolume QMC pareado com incerteza, cardinalidade igual hierárquica e testes pareados. Ausência de significância nunca é tratada como equivalência.

In [ ]:
from pathlib import Path
import json, os, time, math, gc
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize, minimize_scalar
from scipy.spatial import ConvexHull, Delaunay, cKDTree
from scipy.stats import qmc

def project_root(start=Path.cwd()):
    p=start.resolve()
    for candidate in (p,*p.parents):
        if (candidate/'configs'/'smoke.json').exists(): return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada')

ROOT=project_root(); MODE=os.environ.get('CNBI_MODE','SMOKE').upper()
CFG=json.loads((ROOT/'configs'/f'{MODE.lower()}.json').read_text(encoding='utf-8'))
for key in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS'): os.environ[key]='1'
ALPHA=2**0.75; DELTA_BY_K={2:.10,3:.10,4:.20,5:.50}

from scipy.spatial import cKDTree
from scipy.stats import friedmanchisquare,wilcoxon
from statsmodels.stats.multitest import multipletests
TAB=ROOT/'results'/'tables'; TAB.mkdir(parents=True,exist_ok=True)

def nearest(A,B): return cKDTree(B).query(A,k=1)[0]
def GD(A,R): return float(nearest(A,R).mean())
def IGD(A,R): return float(nearest(R,A).mean())
def spacing(A):
    d=cKDTree(A).query(A,k=2)[0][:,1]; return float(np.sqrt(np.mean((d-d.mean())**2)))
def sparsity(A): return float(cKDTree(A).query(A,k=2)[0][:,1].mean())
def hv_qmc(A,m,seed=2026,n=5000,ref=1.1,chunk=1000):
    Z=qmc.Sobol(m,scramble=True,seed=seed).random(n)*ref; hit=0
    for i in range(0,n,chunk): hit+=np.any(np.all(Z[i:i+chunk,None,:]>=A[None,:,:],axis=2),axis=1).sum()
    p=hit/n; vol=ref**m; return p*vol,vol*np.sqrt(max(p*(1-p),0)/n)
def hierarchical_equal_cardinality(F,n):
    from sklearn.cluster import AgglomerativeClustering
    lab=AgglomerativeClustering(n_clusters=n,linkage='ward').fit_predict(F); idx=[]
    for k in range(n):
        members=np.where(lab==k)[0]; center=F[members].mean(0); idx.append(members[np.argmin(np.linalg.norm(F[members]-center,axis=1))])
    return F[np.sort(idx)]
def paired_stats(df,metric):
    wide=df.pivot(index=['scenario','seed'],columns='method',values=metric).dropna(); methods=list(wide)
    if len(methods)<3 or len(wide)<3:return pd.DataFrame()
    fr=friedmanchisquare(*[wide[m] for m in methods]); rows=[]
    for m in methods:
        if m=='CNBI':continue
        difference=(wide['CNBI']-wide[m]) if metric=='HV' else (wide[m]-wide['CNBI']); nonzero=difference[difference!=0]; w=wilcoxon(difference,zero_method='wilcox'); ranks=stats.rankdata(np.abs(nonzero)); total=float(ranks.sum()); effect=0.0 if total==0 else float((ranks[nonzero>0].sum()-ranks[nonzero<0].sum())/total); rows.append({'method':m,'p_raw':w.pvalue,'effect_rank_biserial':effect,'effect_positive_favors':'CNBI','zero_differences':int((difference==0).sum())})
    out=pd.DataFrame(rows); out['p_holm']=multipletests(out.p_raw,method='holm')[1]; out['friedman_p']=fr.pvalue; return out

# Metric self-tests, including paired QMC uncertainty.
rng=np.random.default_rng(7); R=rng.random((300,4)); A=R[:50]; assert GD(A,R)==0 and IGD(A,R)>=0 and spacing(A)>=0 and sparsity(A)>=0
hv,se=hv_qmc(A,4,n=int(CFG['hv_samples'])); assert hv>=0 and se>=0
pd.DataFrame([{'mode':MODE,'GD_self':GD(A,R),'IGD_self':IGD(A,R),'HV_self':hv,'HV_se':se}]).to_csv(TAB/'smoke_metrics.csv',index=False)
print('Métricas SMOKE OK:',{'HV':hv,'SE':se})


## Análise permanente dos resultados reais

Avalia todas as soluções no ground truth somente após a otimização, calcula métricas completas e equalizadas e produz inferência pareada por cenário.

In [ ]:
# PERMANENT REAL RESULT ANALYSIS
def analyze_real_results():
    if not CFG['run_optimizers']: return pd.DataFrame()
    det=pd.read_csv(TAB/f'{MODE.lower()}_deterministic_runs.csv'); ea=pd.read_csv(TAB/f'{MODE.lower()}_ea_runs.csv'); runs=pd.concat([det,ea],ignore_index=True)
    runs.to_csv(TAB/f'{MODE.lower()}_method_runs.csv',index=False)
    metrics=[]; fronts={}
    for _,r in runs.iterrows():
        if r.status not in {'COMPLETED','COMPLETED_WITH_INFEASIBLE_SUBPROBLEMS'}: continue
        dat=np.load(ROOT/r.checkpoint); X=np.asarray(dat['X'],float); scenario=r.scenario; A=np.load(ROOT/'data'/'generated'/f'{scenario}_scenario.npz')['anchors']; F=np.sum((X[:,None,:]-A[None,:,:])**2,axis=2)
        ref=np.load(ROOT/'data'/'reference_fronts'/f'{scenario}_pareto_reference.npz')['F']; bundle=np.load(ROOT/'data'/'reference_fronts'/f'{scenario}_pareto_reference.npz'); ideal=np.asarray(bundle['ideal_true']); amp=np.maximum(np.asarray(bundle['nadir_true'])-ideal,1e-12); Fn=(F-ideal)/amp; Rn=(ref-ideal)/amp
        key=(scenario,int(r.seed),r.method); fronts[key]=(Fn,Rn)
        hv,hse=hv_qmc(Fn,Fn.shape[1],seed=2026+int(r.seed),n=int(CFG['hv_samples']))
        metrics.append({'scenario':scenario,'seed':int(r.seed),'method':r.method,'comparison':'complete','n':len(Fn),'GD':GD(Fn,Rn),'IGD':IGD(Fn,Rn),'HV':hv,'HV_se':hse,'Spacing':spacing(Fn),'Sparsity':sparsity(Fn)})
    complete=pd.DataFrame(metrics); equal=[]
    for (scenario,seed),g in complete.groupby(['scenario','seed']):
        target=int(g.n.min())
        for _,r in g.iterrows():
            F,R=fronts[(scenario,int(seed),r.method)]; Fe=hierarchical_equal_cardinality(F,target) if len(F)>target else F; hv,hse=hv_qmc(Fe,Fe.shape[1],seed=2026+int(seed),n=int(CFG['hv_samples']))
            equal.append({'scenario':scenario,'seed':int(seed),'method':r.method,'comparison':'equal_cardinality','n':len(Fe),'GD':GD(Fe,R),'IGD':IGD(Fe,R),'HV':hv,'HV_se':hse,'Spacing':spacing(Fe),'Sparsity':sparsity(Fe)})
    out=pd.concat([complete,pd.DataFrame(equal)],ignore_index=True); out.to_csv(TAB/f'{MODE.lower()}_metrics.csv',index=False)
    stats_rows=[]
    for (scenario,comparison),d in out.groupby(['scenario','comparison']):
      for metric in ('GD','IGD','HV','Spacing','Sparsity'):
        s=paired_stats(d.rename(columns={'method':'method'}),metric)
        if not s.empty: s.insert(0,'metric',metric); s.insert(0,'comparison',comparison); s.insert(0,'scenario',scenario); stats_rows.append(s)
    stats_out=pd.concat(stats_rows,ignore_index=True) if stats_rows else pd.DataFrame(); stats_out.to_csv(TAB/f'{MODE.lower()}_statistics.csv',index=False)
    total_disk=sum(x.stat().st_size for base in (ROOT/'data',ROOT/'results') for x in base.rglob('*') if x.is_file())
    resource={'wall_seconds':float(runs.wall_seconds.sum()),'cpu_seconds':float(runs.get('cpu_seconds',pd.Series(0,index=runs.index)).sum()),'peak_rss_bytes':int(os.environ.get('CNBI_PEAK_RSS_BYTES','0')),'disk_bytes':int(total_disk)}
    (TAB/f'{MODE.lower()}_resource_summary.json').write_text(json.dumps(resource,indent=2),encoding='utf-8')
    return out

real_metrics=pd.DataFrame()  # execução adiada; a redução canônica ocorre uma única vez na célula auditável abaixo


## Análise final auditável

Filtra factibilidade e não dominância, mantém amostras de HV pareadas, agrega custo/convergência e registra explicitamente insuficiência de blocos estatísticos no PILOT.

In [ ]:
# FILTERED COST-AWARE RESULT ANALYSIS
def nondominated(F):
    F=np.asarray(F,float); keep=np.ones(len(F),bool)
    for i in range(len(F)):
        if keep[i]: keep[i]=not np.any(np.all(F<=F[i],axis=1)&np.any(F<F[i],axis=1))
    return keep

def analyze_filtered_results():
    if not CFG['run_optimizers']: return pd.DataFrame()
    det=pd.read_csv(TAB/f'{MODE.lower()}_deterministic_runs.csv'); ea=pd.read_csv(TAB/f'{MODE.lower()}_ea_runs.csv'); runs=pd.concat([det,ea],ignore_index=True); runs.to_csv(TAB/f'{MODE.lower()}_method_runs.csv',index=False)
    rows=[]; fronts={}
    for _,r in runs.iterrows():
      if r.status not in {'COMPLETED','COMPLETED_WITH_INFEASIBLE_SUBPROBLEMS'}: continue
      dat=np.load(ROOT/r.checkpoint,allow_pickle=False); Xv=np.asarray(dat['X'],float); success=np.asarray(dat['success'],bool); feasible=np.sum(Xv*Xv,axis=1)<=ALPHA**2+1e-8; valid=success&feasible; A=np.load(ROOT/'data'/'generated'/f'{r.scenario}_scenario.npz')['anchors']; F=np.sum((Xv[:,None,:]-A[None,:,:])**2,axis=2); F=F[valid]; F=F[nondominated(F)]
      ref=np.load(ROOT/'data'/'reference_fronts'/f'{r.scenario}_pareto_reference.npz')['F']; bundle=np.load(ROOT/'data'/'reference_fronts'/f'{r.scenario}_pareto_reference.npz'); ideal=np.asarray(bundle['ideal_true']); amp=np.maximum(np.asarray(bundle['nadir_true'])-ideal,1e-12); Fn=(F-ideal)/amp; Rn=(ref-ideal)/amp; key=(r.scenario,int(r.seed),r.method); fronts[key]=(Fn,Rn)
      hv,hse=hv_qmc(Fn,Fn.shape[1],seed=2026+int(r.seed),n=int(CFG['hv_samples'])); rows.append({'scenario':r.scenario,'seed':int(r.seed),'method':r.method,'comparison':'complete','n':len(Fn),'returned_n':len(Xv),'GD':GD(Fn,Rn),'IGD':IGD(Fn,Rn),'HV':hv,'HV_se':hse,'Spacing':spacing(Fn),'Sparsity':sparsity(Fn),'feasible_fraction':float(feasible.mean()) if len(feasible) else 0.,'converged_fraction':float(success.mean()) if len(success) else 0.,'rsm_evaluations':int(r.rsm_evaluations),'wall_seconds':float(r.wall_seconds),'cpu_seconds':float(r.cpu_seconds)})
    complete=pd.DataFrame(rows); equal=[]
    for (scenario,seed),g in complete.groupby(['scenario','seed']):
      target=int(g.n.min())
      for _,r in g.iterrows():
        F,R=fronts[(scenario,int(seed),r.method)]; Fe=hierarchical_equal_cardinality(F,target) if len(F)>target else F; hv,hse=hv_qmc(Fe,Fe.shape[1],seed=2026+int(seed),n=int(CFG['hv_samples'])); equal.append({**r.to_dict(),'comparison':'equal_cardinality','n':len(Fe),'GD':GD(Fe,R),'IGD':IGD(Fe,R),'HV':hv,'HV_se':hse,'Spacing':spacing(Fe),'Sparsity':sparsity(Fe)})
    out=pd.concat([complete,pd.DataFrame(equal)],ignore_index=True); out.to_csv(TAB/f'{MODE.lower()}_metrics.csv',index=False)
    summary=out.groupby(['scenario','method','comparison'],as_index=False).agg(**{f'{metric}_median':(metric,'median') for metric in ('GD','IGD','HV','Spacing','Sparsity')},**{f'{metric}_iqr':(metric,lambda x:x.quantile(.75)-x.quantile(.25)) for metric in ('GD','IGD','HV','Spacing','Sparsity')}); summary.to_csv(TAB/f'{MODE.lower()}_summary.csv',index=False)
    rank_rows=[]
    for (scenario,comparison,seed),g in out.groupby(['scenario','comparison','seed']):
      for metric in ('GD','IGD','HV','Spacing','Sparsity'):
        ascending=metric!='HV'; ranks=g[metric].rank(method='average',ascending=ascending)
        for method,rank in zip(g.method,ranks): rank_rows.append({'scenario':scenario,'comparison':comparison,'seed':seed,'metric':metric,'method':method,'rank':rank})
    pd.DataFrame(rank_rows).to_csv(TAB/f'{MODE.lower()}_rankings.csv',index=False)
    stats_rows=[]
    for (scenario,comparison),d in out.groupby(['scenario','comparison']):
      for metric in ('GD','IGD','HV','Spacing','Sparsity'):
        if d.seed.nunique()<3: stats_rows.append({'scenario':scenario,'comparison':comparison,'metric':metric,'test':'Friedman/Wilcoxon-Holm','status':'INSUFFICIENT_BLOCKS','n_blocks':d.seed.nunique()}); continue
        stat=paired_stats(d,metric)
        if stat.empty: stats_rows.append({'scenario':scenario,'comparison':comparison,'metric':metric,'test':'Friedman/Wilcoxon-Holm','status':'NO_VALID_COMMON_BLOCK','n_blocks':d.seed.nunique()})
        else:
          stat.insert(0,'status','COMPLETED'); stat.insert(0,'metric',metric); stat.insert(0,'comparison',comparison); stat.insert(0,'scenario',scenario); stats_rows.extend(stat.to_dict('records'))
    pd.DataFrame(stats_rows).to_csv(TAB/f'{MODE.lower()}_statistics.csv',index=False)
    return out

filtered_metrics=analyze_filtered_results()
if CFG['run_optimizers']: print(f'Métricas auditáveis: {len(filtered_metrics)} linhas')